In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count


def main():
    spark = (
        SparkSession.builder
        .appName("read_gold_dim_categories")
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .getOrCreate()
    )

    gold_path = "file:/data/gold/dim_categories"

    print("Reading Gold dimension: dim_categories")
    print(f"Path: {gold_path}")

    df = spark.read.parquet(gold_path)

    print("\nSchema:")
    df.printSchema()

    print("\nSample data:")
    df.show(truncate=False)

    print("\nCurrent records only (is_current = true):")
    df_current = df.filter(col("is_current") == True)
    df_current.show(truncate=False)

    print("\nVersions per category:")
    (
        df.groupBy("category_code")
        .agg(count("*").alias("versions"))
        .orderBy(col("versions").desc())
        .show()
    )

    print("\nCreating temp view: dim_categories")
    df.createOrReplaceTempView("dim_categories")

    print("\nQuerying current dimension records:")
    spark.sql("""
        SELECT
            category_sk,
            category_code,
            category_description,
            valid_from,
            valid_to
        FROM dim_categories
        WHERE is_current = true
        ORDER BY category_code
    """).show(truncate=False)

    spark.stop()


if __name__ == "__main__":
    main()


Reading Gold dimension: dim_categories
Path: file:/data/gold/dim_categories

Schema:
root
 |-- category_sk: long (nullable = true)
 |-- category_code: integer (nullable = true)
 |-- category_description: string (nullable = true)
 |-- valid_from: date (nullable = true)
 |-- valid_to: date (nullable = true)
 |-- is_current: boolean (nullable = true)


Sample data:
+-----------+-------------+--------------------+----------+----------+----------+
|category_sk|category_code|category_description|valid_from|valid_to  |is_current|
+-----------+-------------+--------------------+----------+----------+----------+
|0          |1            |Physical Books      |2026-02-10|2026-02-10|false     |
|1          |2            |Cell Phones         |2026-02-10|NULL      |true      |
|2          |3            |Tablets             |2026-02-10|NULL      |true      |
|3          |4            |Notebooks           |2026-02-10|NULL      |true      |
|4          |5            |Office Supply       |2026-02-10|NU